In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table("crude_ops.silver.drilling_enriched")

# Adding a time window (15 minutes) to see changes over time
window_spec = Window.partitionBy("phase", "operation")

gold_df = silver_df.withColumn(
    "GR_cleaned", 
    F.when(F.col("GR") <= 0, F.lit(None)).otherwise(F.col("GR"))
).withColumn(
    "avg_GR_global", F.avg("GR_cleaned").over(window_spec)
).withColumn(
    "stddev_GR_global", F.stddev("GR_cleaned").over(window_spec)
).withColumn(
    # Z-score logic: (value - mean) / stddev
    "z_score", 
    (F.col("GR_cleaned") - F.col("avg_GR_global")) / F.col("stddev_GR_global")
).withColumn(
    # Маркуємо аномалії (Z-score > 3 або < -3)
    "is_anomaly", 
    F.when(F.abs(F.col("z_score")) > 3, 1).otherwise(0)
)

# 3. Тепер групуємо, як ти і робив, але додаємо кількість аномалій у вікно
final_gold_df = gold_df.groupBy(
    "phase", 
    "operation", 
    F.window("event_timestamp", "1 seconds")
).agg(
    F.avg("DEPTH").alias("avg_depth"),
    F.avg("GR_cleaned").alias("avg_gamma_ray"),
    F.max("is_anomaly").alias("anomaly_detected"), # Чи була хоч одна аномалія в цю секунду
    F.count("*").alias("data_points_count")
).select(
    "window.start", 
    "phase", 
    "operation", 
    "avg_depth", 
    "avg_gamma_ray", 
    "anomaly_detected",
    "data_points_count"
).orderBy("start")

final_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crude_ops.gold.drilling_stats_summary")

display(spark.table("crude_ops.gold.drilling_stats_summary"))

In [0]:
%skip
%sql
select * from crude_ops.gold.drilling_stats_summary
order by anomaly_detected desc
limit 5

In [0]:
%skip
%sql
select distinct (event_timestamp)
from crude_ops.silver.drilling_enriched 
--where phase = 'drilling'


In [0]:
%skip
%sql
select * from crude_ops.bronze.drilling_raw